# Data Preparation & Cleaning: Java Code Vulnerability Dataset

This notebook performs rigorous Data Cleaning and Exploratory Data Analysis (EDA) on both the vulnerable Kaggle dataset and the clean CodeSearchNet dataset before merging them into a final instruction-tuned dataset.

In [10]:
from datasets import disable_progress_bar
disable_progress_bar()
import pandas as pd
import json
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Analyze the Vulnerable Dataset (Kaggle)

In [11]:
print("Loading Kaggle dataset...")
df_vuln = pd.read_csv("vulnerability_fix_dataset.csv")

print(f"Original Shape: {df_vuln.shape}")
print("\n--- Missing Values ---")
print(df_vuln.isnull().sum())
print("\n--- First 3 Rows ---")
display(df_vuln.head(3))

Loading Kaggle dataset...
Original Shape: (35000, 3)

--- Missing Values ---
vulnerability_type    0
vulnerable_code       0
fixed_code            0
dtype: int64

--- First 3 Rows ---


,vulnerability_type,vulnerable_code,fixed_code
0,SQL Injection,import java.sql.*;\n\npublic class SQLInjectio...,import java.sql.*;\n\npublic class SQLInjectio...
1,SQL Injection,import java.sql.*;\n\npublic class VulnerableS...,import java.sql.*;\n\npublic class SecureSQLIn...
2,SQL Injection,import java.sql.*;\n\npublic class SQLInjectio...,To prevent SQL Injection attacks in the provid...


### Clean Kaggle Dataset
We will drop any rows with missing values and remove exact duplicates.

In [12]:
df_vuln = df_vuln.dropna()
df_vuln = df_vuln.drop_duplicates()
print(f"Original Kaggle Dataset Shape: {df_vuln.shape}")

# Filter out placeholder rows ('No response generated')
def is_vuln_placeholder(row):
    vuln = str(row['vulnerable_code']).lower()
    fixed = str(row['fixed_code']).lower()
    return "no response generated" in vuln or "no response generated" in fixed

df_vuln = df_vuln[~df_vuln.apply(is_vuln_placeholder, axis=1)]
print(f"After placeholder filter: {df_vuln.shape}")

# Filter out truncated rows
def is_vuln_truncated(row):
    fixed = str(row['fixed_code']).strip()
    if "```java" in fixed:
        parts = fixed.split("```java")
        if len(parts) >= 2 and "```" not in parts[1]:
            return True
    return False

df_vuln = df_vuln[~df_vuln.apply(is_vuln_truncated, axis=1)].copy()
print(f"After truncation filter (Final Clean Vuln): {df_vuln.shape}")
num_vuln_samples = len(df_vuln)


Original Kaggle Dataset Shape: (32483, 3)
After placeholder filter: (32234, 3)
After truncation filter (Final Clean Vuln): (20197, 3)


## 2. Load and Analyze Clean Dataset (Hugging Face CodeSearchNet)
CodeSearchNet contains safe, open-source Java functions. We will load it directly into a Pandas DataFrame for cleaning.

In [13]:
print("Downloading CodeSearchNet Java dataset from Hugging Face...")
hf_dataset = load_dataset('Nan-Do/code-search-net-java', split='train')

print("Converting to Pandas DataFrame for analysis...")
df_hf = hf_dataset.to_pandas()

print(f"Original Shape: {df_hf.shape}")
print("\n--- Missing Values ---")
print(df_hf.isnull().sum())
print("\n--- First 3 Rows ---")
display(df_hf.head(3))

Converting to Pandas DataFrame for analysis...
Original Shape: (495953, 13)

--- Missing Values ---
repo                0
path                0
func_name           0
original_string     0
language            0
code                0
code_tokens         0
docstring           0
docstring_tokens    0
sha                 0
url                 0
partition           0
summary             0
dtype: int64

--- First 3 Rows ---


,repo,path,func_name,original_string,language,code,code_tokens,docstring,docstring_tokens,sha,url,partition,summary
0,spring-projects/spring-boot,spring-boot-project/spring-boot/src/main/java/...,IndexedElementsBinder.bindIndexed,protected final void bindIndexed(Configuration...,java,protected final void bindIndexed(Configuration...,"[protected, final, void, bindIndexed, (, Confi...",Bind indexed elements to the supplied collecti...,"[Bind, indexed, elements, to, the, supplied, c...",0b27f7c70e164b2b1a96477f1d9c1acba56790c1,https://github.com/spring-projects/spring-boot...,train,Binds the indexed elements of the configuratio...
1,spring-projects/spring-boot,spring-boot-project/spring-boot/src/main/java/...,AbstractFilterRegistrationBean.setServletRegis...,public void setServletRegistrationBeans(\n\t\t...,java,public void setServletRegistrationBeans(\n\t\t...,"[public, void, setServletRegistrationBeans, (,...",Set {@link ServletRegistrationBean}s that the ...,"[Set, {]",0b27f7c70e164b2b1a96477f1d9c1acba56790c1,https://github.com/spring-projects/spring-boot...,train,Sets the servlet registration beans.
2,spring-projects/spring-boot,spring-boot-project/spring-boot/src/main/java/...,AbstractFilterRegistrationBean.addServletRegis...,public void addServletRegistrationBeans(\n\t\t...,java,public void addServletRegistrationBeans(\n\t\t...,"[public, void, addServletRegistrationBeans, (,...",Add {@link ServletRegistrationBean}s for the f...,"[Add, {]",0b27f7c70e164b2b1a96477f1d9c1acba56790c1,https://github.com/spring-projects/spring-boot...,train,Add servlet registration beans.


### Clean CodeSearchNet Dataset
We need to ensure there are no nulls in the code columns and drop duplicates before we sample from it.

In [14]:
# Determine the code column
code_col = 'original_string' if 'original_string' in df_hf.columns else 'func_code_string'

df_hf = df_hf.dropna(subset=[code_col])
df_hf = df_hf.drop_duplicates(subset=[code_col])
print(f"Cleaned CodeSearchNet Dataset Shape: {df_hf.shape}")

Cleaned CodeSearchNet Dataset Shape: (495953, 13)


## 3. Sample and Format Clean Data to Match Schema
We will sample exactly the same number of rows as our cleaned Kaggle dataset to get a 1:1 mix. The vulnerability type will be labeled as `Safe`.

In [15]:
df_clean = df_hf.sample(n=num_vuln_samples, random_state=42).copy()

df_clean['vulnerability_type'] = 'Safe'
df_clean['vulnerable_code'] = df_clean[code_col]
df_clean['fixed_code'] = df_clean[code_col]

# Keep only the required columns
df_clean = df_clean[['vulnerability_type', 'vulnerable_code', 'fixed_code']]
print(f"Formatted Clean Dataset Shape: {df_clean.shape}")
display(df_clean.head(2))

Formatted Clean Dataset Shape: (20197, 3)


,vulnerability_type,vulnerable_code,fixed_code
388298,Safe,"public static void mult(DMatrixSparseCSC A, DM...","public static void mult(DMatrixSparseCSC A, DM..."
233532,Safe,public Interval withPeriodAfterStart(ReadableP...,public Interval withPeriodAfterStart(ReadableP...


## 4. Merge, Shuffle, and Format for LLM Instruction Tuning
We concatenate the two DataFrames, shuffle them thoroughly, and convert them to an instruction schema (`instruction`, `input`, `output`).

In [16]:
df_combined = pd.concat([df_vuln, df_clean], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Total dataset size after merging: {len(df_combined)}")

instruction = "Analyze the following Java code. If a vulnerability exists, provide the fixed code. If it is safe, output the original code."
df_combined['instruction'] = instruction

# Rename columns to standard instruction-tuning format
df_jsonl = df_combined.rename(columns={
    'vulnerable_code': 'input',
    'fixed_code': 'output'
})[['instruction', 'input', 'output', 'vulnerability_type']]

display(df_jsonl.head(3))

Total dataset size after merging: 40394


,instruction,input,output,vulnerability_type
0,Analyze the following Java code. If a vulnerab...,@Override\n\tpublic Date decode(String value) ...,@Override\n\tpublic Date decode(String value) ...,Safe
1,Analyze the following Java code. If a vulnerab...,@EventThread\n protected void clientSession...,@EventThread\n protected void clientSession...,Safe
2,Analyze the following Java code. If a vulnerab...,public boolean hasRoleForResource(CmsRequestCo...,public boolean hasRoleForResource(CmsRequestCo...,Safe


## 5. Pre-Training Checks: Class Distribution & Sequence Lengths
Before training a model, we must verify two critical things:
1. **Class Distribution**: Ensure the percentage of Safe vs Vulnerable data is balanced. If we have too much vulnerable data, the model will over-trigger. We also check the distribution of specific bug types.
2. **Sequence Lengths**: LLMs have strict context windows (e.g., 2048, 4096 tokens). If our code snippets are too long, they will be truncated, leading to failed syntax generation or Out-Of-Memory (OOM) errors during training. We calculate string lengths to help choose the `max_seq_length` hyperparameter.

In [17]:
print("=== CLASS DISTRIBUTION ===")
total_rows = len(df_jsonl)
safe_count = len(df_jsonl[df_jsonl['vulnerability_type'] == 'Safe'])
vuln_count = total_rows - safe_count

print(f"Safe Data: {safe_count} rows ({safe_count/total_rows*100:.2f}%)")
print(f"Vulnerable Data: {vuln_count} rows ({vuln_count/total_rows*100:.2f}%)\n")

print("Detailed Bug Type Distribution:")
print(df_jsonl['vulnerability_type'].value_counts(normalize=True) * 100)

print("\n=== SEQUENCE LENGTH CHECKS ===")
# Calculate approximate lengths (in characters). 
# Note: 1 token is roughly ~4 characters for typical code.
df_jsonl['input_char_len'] = df_jsonl['input'].astype(str).apply(len)
df_jsonl['output_char_len'] = df_jsonl['output'].astype(str).apply(len)

print(f"Input Lengths - Median: {df_jsonl['input_char_len'].median():.0f} chars, Max: {df_jsonl['input_char_len'].max()} chars")
print(f"Output Lengths - Median: {df_jsonl['output_char_len'].median():.0f} chars, Max: {df_jsonl['output_char_len'].max()} chars")
print("\nRecommendation: When fine-tuning (e.g. LoRA), set `max_seq_length` to accommodate the 95th percentile of your combined input+output lengths to avoid truncation. If the max length is very high (e.g., > 8000 chars / ~2000 tokens), you may need to filter out those outliers or use a model with a larger context window.")

=== CLASS DISTRIBUTION ===
Safe Data: 20197 rows (50.00%)
Vulnerable Data: 20197 rows (50.00%)

Detailed Bug Type Distribution:
vulnerability_type
Safe                          50.000000
Cross-Site Scripting (XSS)    19.626677
SQL Injection                 15.819181
Buffer Overflow                6.986186
Command Injection              6.723771
Path Traversal                 0.643660
Insecure Deserialization       0.200525
Name: proportion, dtype: float64

=== SEQUENCE LENGTH CHECKS ===
Input Lengths - Median: 570 chars, Max: 280864 chars
Output Lengths - Median: 677 chars, Max: 280864 chars

Recommendation: When fine-tuning (e.g. LoRA), set `max_seq_length` to accommodate the 95th percentile of your combined input+output lengths to avoid truncation. If the max length is very high (e.g., > 8000 chars / ~2000 tokens), you may need to filter out those outliers or use a model with a larger context window.


## 6. Split and Export (with Truncation Filter)
Before exporting, we filter out any truncated/corrupted outputs to ensure the model trains only on complete code corrections.

In [18]:
# Drop temporary helper columns
df_jsonl = df_jsonl.drop(columns=['input_char_len', 'output_char_len'])

print(f"Final clean 50-50 dataset size: {len(df_jsonl)}")
print(f"\n--- FINAL CLEAN DATASET DISTRIBUTION ---")
print(df_jsonl['vulnerability_type'].value_counts())

# Split into Train (85%), Validation (10%), and Test (5%)
train_df, temp_df = train_test_split(df_jsonl, test_size=0.15, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=(1/3), random_state=42)

print(f"\n--- EXPORT SPLITS ---")
print(f"Train size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")
print(f"Test size:  {len(test_df)}")

train_df.to_json("train.jsonl", orient="records", lines=True)
val_df.to_json("val.jsonl", orient="records", lines=True)
test_df.to_json("test.jsonl", orient="records", lines=True)

print("\n✅ Export successful: train.jsonl, val.jsonl, test.jsonl created.")

Final clean 50-50 dataset size: 40394

--- FINAL CLEAN DATASET DISTRIBUTION ---
vulnerability_type
Safe                          20197
Cross-Site Scripting (XSS)     7928
SQL Injection                  6390
Buffer Overflow                2822
Command Injection              2716
Path Traversal                  260
Insecure Deserialization         81
Name: count, dtype: int64

--- EXPORT SPLITS ---
Train size: 34334
Val size:   4040
Test size:  2020

✅ Export successful: train.jsonl, val.jsonl, test.jsonl created.


## 7. Diagnostic Check: Count Truncated/Incomplete Outputs
Verify the new cleaned dataset has 0 truncated examples.

In [19]:
import json

def verify_split(filename):
    truncated_count = 0
    placeholder_count = 0
    total_count = 0
    
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            total_count += 1
            data = json.loads(line)
            inp = (data.get('input', '') or '').strip()
            out = (data.get('output', '') or '').strip()
            
            # Check for placeholders
            if "no response generated" in inp.lower() or "no response generated" in out.lower():
                placeholder_count += 1
                
            # Check for truncation
            if "```java" in out:
                parts = out.split("```java")
                if len(parts) >= 2 and "```" not in parts[1]:
                    truncated_count += 1
                    
    print(f"File: {filename}")
    print(f"  Total Checked: {total_count}")
    print(f"  Placeholders:  {placeholder_count} (Expected: 0) -> {'PASSED' if placeholder_count == 0 else 'FAILED'}")
    print(f"  Truncated:     {truncated_count} (Expected: 0) -> {'PASSED' if truncated_count == 0 else 'FAILED'}")
    print()

print("=== COMPREHENSIVE SPLIT VERIFICATION ===")
verify_split("train.jsonl")
verify_split("val.jsonl")
verify_split("test.jsonl")


=== COMPREHENSIVE SPLIT VERIFICATION ===
File: train.jsonl
  Total Checked: 34334
  Placeholders:  0 (Expected: 0) -> PASSED
  Truncated:     0 (Expected: 0) -> PASSED

File: val.jsonl
  Total Checked: 4040
  Placeholders:  0 (Expected: 0) -> PASSED
  Truncated:     0 (Expected: 0) -> PASSED

File: test.jsonl
  Total Checked: 2020
  Placeholders:  0 (Expected: 0) -> PASSED
  Truncated:     0 (Expected: 0) -> PASSED

